In [19]:
import pandas as pd
import numpy as np
from collections import defaultdict, deque
from pathlib import Path


# ============================================================
# SETTINGS
# ============================================================

FILLS_FILE = "GDU_fills_converted.xlsx"
TS_FILE = "GDUTS.xlsx"
GC_FILE = "GC OHLC 10sec.xlsx"

OUTPUT_FILE = "GDU_Analysis_FIFO_GC_Ranges_DivideBy0.1.xlsx"

FILLS_SHEET = "Sheet1"
TS_SHEET = "Sheet1"

GC_TICK_SIZE = 0.1
PNL_MULTIPLIER = 1000


# ============================================================
# HELPERS
# ============================================================

def normalize_side(value):
    """Convert side into B or S."""
    value = str(value).strip().upper()

    if value in ["B", "BUY"]:
        return "B"

    if value in ["S", "SELL"]:
        return "S"

    return None


def make_datetime(date_col, time_col):
    """
    Combine separate Date and Time columns into one timestamp.
    """
    date_text = pd.to_datetime(date_col).dt.strftime("%Y-%m-%d")
    time_text = time_col.astype(str)

    return pd.to_datetime(
        date_text + " " + time_text,
        errors="coerce"
    )


# ============================================================
# LOAD FILLS
# ============================================================

def load_fills(path):
    df = pd.read_excel(path, sheet_name=FILLS_SHEET)

    print("Fill columns:")
    print(df.columns.tolist())

    # Expected structure:
    # Date
    # Time
    # ...
    # Contract
    # B/S
    # Lots
    # Price

    df = df.copy()

    df["DateTime"] = make_datetime(
        df.iloc[:, 0],
        df.iloc[:, 1]
    )

    df["Contract"] = df.iloc[:, 3].astype(str).str.strip()

    df["Side"] = df.iloc[:, 4].apply(normalize_side)

    df["Lots"] = pd.to_numeric(
        df.iloc[:, 5],
        errors="coerce"
    )

    df["Price"] = pd.to_numeric(
        df.iloc[:, 6],
        errors="coerce"
    )

    df = df.dropna(
        subset=[
            "DateTime",
            "Contract",
            "Side",
            "Lots",
            "Price"
        ]
    )

    df = df[df["Lots"] > 0]

    # Important: stable sort so same-timestamp fills
    # retain their original Excel order.
    df["_OriginalOrder"] = np.arange(len(df))

    df = df.sort_values(
        ["DateTime", "_OriginalOrder"],
        kind="stable"
    ).reset_index(drop=True)

    return df


# ============================================================
# STRICT FIFO MATCHING
# ============================================================

def fifo_match(fills):
    """
    Strict FIFO independently for each contract.

    A buy offsets the oldest open sell.
    A sell offsets the oldest open buy.

    Supports partial fills.
    """

    open_buys = defaultdict(deque)
    open_sells = defaultdict(deque)

    entries = []

    for _, row in fills.iterrows():

        contract = row["Contract"]
        side = row["Side"]
        qty = float(row["Lots"])

        fill = {
            "datetime": row["DateTime"],
            "contract": contract,
            "side": side,
            "price": float(row["Price"]),
            "qty": qty
        }

        remaining = qty

        if side == "B":
            opposite = open_sells[contract]
            own = open_buys[contract]

        else:
            opposite = open_buys[contract]
            own = open_sells[contract]

        # ----------------------------------------------------
        # Close existing opposite FIFO positions
        # ----------------------------------------------------

        while remaining > 0 and opposite:

            entry = opposite[0]

            matched_qty = min(
                remaining,
                entry["remaining"]
            )

            entry["exits"].append({
                "datetime": fill["datetime"],
                "price": fill["price"],
                "qty": matched_qty
            })

            entry["remaining"] -= matched_qty
            remaining -= matched_qty

            if entry["remaining"] <= 1e-12:
                opposite.popleft()

        # ----------------------------------------------------
        # Remaining quantity becomes new open position
        # ----------------------------------------------------

        if remaining > 1e-12:

            entry = {
                "entry_datetime": fill["datetime"],
                "contract": contract,
                "side": side,
                "entry_price": fill["price"],
                "original_qty": remaining,
                "remaining": remaining,
                "exits": []
            }

            entries.append(entry)
            own.append(entry)

    # ========================================================
    # CONVERT FIFO ENTRIES INTO MATCHED TRADES
    # ========================================================

    trades = []

    for entry in entries:

        matched_qty = sum(
            x["qty"]
            for x in entry["exits"]
        )

        if matched_qty <= 0:
            continue

        weighted_exit_price = (
            sum(
                x["price"] * x["qty"]
                for x in entry["exits"]
            )
            / matched_qty
        )

        exit_datetime = max(
            x["datetime"]
            for x in entry["exits"]
        )

        if entry["side"] == "B":

            pnl = (
                weighted_exit_price
                - entry["entry_price"]
            ) * matched_qty * PNL_MULTIPLIER

        else:

            pnl = (
                entry["entry_price"]
                - weighted_exit_price
            ) * matched_qty * PNL_MULTIPLIER

        duration_ms = (
            exit_datetime
            - entry["entry_datetime"]
        ).total_seconds() * 1000

        trades.append({
            "Entry Time": entry["entry_datetime"],
            "Exit Time": exit_datetime,
            "Contract": entry["contract"],
            "Entry Side": entry["side"],
            "Lots": matched_qty,
            "Entry Price": entry["entry_price"],
            "Weighted Exit Price": weighted_exit_price,
            "PnL": pnl,
            "Duration (ms)": duration_ms,
            "Exit Fill Count": len(entry["exits"])
        })

    trades = pd.DataFrame(trades)

    trades = trades.sort_values(
        "Entry Time"
    ).reset_index(drop=True)

    return trades


# ============================================================
# LOAD GDU TIME & SALES
# ============================================================

def load_time_sales(path):

    df = pd.read_excel(
        path,
        sheet_name=TS_SHEET
    )

    df = df.copy()

    df["DateTime"] = make_datetime(
        df.iloc[:, 0],
        df.iloc[:, 1]
    )

    df["Contract"] = (
        df.iloc[:, 2]
        .astype(str)
        .str.strip()
    )

    df["Lots"] = pd.to_numeric(
        df.iloc[:, 3],
        errors="coerce"
    )

    df = df.dropna(
        subset=[
            "DateTime",
            "Contract",
            "Lots"
        ]
    )

    return df.sort_values("DateTime")


# ============================================================
# GDU LOT MATCHING
# ============================================================

def find_gdu_lots(
    trade_time,
    contract,
    duration_ms,
    ts
):

    contract_ts = ts[
        ts["Contract"] == contract
    ]

    if contract_ts.empty:
        return np.nan

    # --------------------------------------------------------
    # Trade duration >100ms:
    # sum prints within +/-100ms
    # --------------------------------------------------------

    if duration_ms > 100:

        start = trade_time - pd.Timedelta(
            milliseconds=200
        )

        end = trade_time + pd.Timedelta(
            milliseconds=200
        )

        window = contract_ts[
            (contract_ts["DateTime"] >= start)
            &
            (contract_ts["DateTime"] <= end)
        ]

        if not window.empty:
            return window["Lots"].sum()

    # --------------------------------------------------------
    # Otherwise nearest Time & Sales print
    # --------------------------------------------------------

    diff = (
        contract_ts["DateTime"]
        - trade_time
    ).abs()

    idx = diff.idxmin()

    # Optional safety threshold
    if diff.loc[idx] <= pd.Timedelta(seconds=1):
        return contract_ts.loc[idx, "Lots"]

    return np.nan


# ============================================================
# LOAD GC OHLC
# ============================================================

def load_gc_sheet(path, sheet):

    df = pd.read_excel(
        path,
        sheet_name=sheet
    )

    # IMPORTANT:
    # Use COLUMN NAMES, not column positions.
    #
    # Aug might be:
    # Dates | Open | Close | High | Low | Volume
    #
    # Dec might be:
    # Dates | Open | High | Low | Volume

    df.columns = [
        str(c).strip()
        for c in df.columns
    ]

    required = [
        "Dates",
        "High",
        "Low"
    ]

    for col in required:
        if col not in df.columns:
            raise ValueError(
                f"{sheet}: missing column {col}"
            )

    result = pd.DataFrame()

    result["DateTime"] = pd.to_datetime(
        df["Dates"],
        errors="coerce"
    )

    result["High"] = pd.to_numeric(
        df["High"],
        errors="coerce"
    )

    result["Low"] = pd.to_numeric(
        df["Low"],
        errors="coerce"
    )

    result = result.dropna(
        subset=[
            "DateTime",
            "High",
            "Low"
        ]
    )

    return result.sort_values("DateTime")


# ============================================================
# GC RANGE
# ============================================================

def get_gc_range(
    trade_time,
    gc,
    window_seconds
):
    """
    Uses the FIXED time bucket containing the trade.

    Example trade time = 10:23:28

    10 sec:
        10:23:20 -> 10:23:29.999

    1 min:
        10:23:00 -> 10:23:59.999

    5 min:
        10:20:00 -> 10:24:59.999

    30 min:
        10:00:00 -> 10:29:59.999
    """

    freq = f"{window_seconds}s"

    start = trade_time.floor(freq)

    end = (
        start
        + pd.Timedelta(seconds=window_seconds)
    )

    window = gc[
        (gc["DateTime"] >= start)
        &
        (gc["DateTime"] < end)
    ]

    if window.empty:
        return np.nan

    highest_high = window["High"].max()
    lowest_low = window["Low"].min()

    # ========================================================
    # IMPORTANT:
    # GC movement is divided by 0.1
    # ========================================================

    range_ticks = (
        highest_high
        - lowest_low
    ) / GC_TICK_SIZE

    return range_ticks


# ============================================================
# CHOOSE GC CONTRACT
# ============================================================

def choose_gc_data(
    contract,
    gc_aug,
    gc_dec
):

    contract = str(contract)

    if "Dec26" in contract:
        return gc_dec

    return gc_aug


# ============================================================
# ADD GDU + GC DATA TO TRADES
# ============================================================

def enrich_trades(
    trades,
    ts,
    gc_aug,
    gc_dec
):

    gdu_lots = []

    gc10 = []
    gc1m = []
    gc5m = []
    gc30m = []

    for _, trade in trades.iterrows():

        trade_time = trade["Entry Time"]

        # GDU lots
        gdu = find_gdu_lots(
            trade_time,
            trade["Contract"],
            trade["Duration (ms)"],
            ts
        )

        gdu_lots.append(gdu)

        # Select correct GC month
        gc = choose_gc_data(
            trade["Contract"],
            gc_aug,
            gc_dec
        )

        gc10.append(
            get_gc_range(
                trade_time,
                gc,
                10
            )
        )

        gc1m.append(
            get_gc_range(
                trade_time,
                gc,
                60
            )
        )

        gc5m.append(
            get_gc_range(
                trade_time,
                gc,
                300
            )
        )

        gc30m.append(
            get_gc_range(
                trade_time,
                gc,
                1800
            )
        )

    trades["GDU Lots Traded"] = gdu_lots

    trades[
        "GC 10 Sec Range (Ticks)"
    ] = gc10

    trades[
        "GC 1 Min Range (Ticks)"
    ] = gc1m

    trades[
        "GC 5 Min Range (Ticks)"
    ] = gc5m

    trades[
        "GC 30 Min Range (Ticks)"
    ] = gc30m

    return trades


# ============================================================
# GDU ANALYSIS
# ============================================================

def create_gdu_analysis(trades):

    data = trades.dropna(
        subset=["GDU Lots Traded"]
    ).copy()

    data["GDU Lots Traded"] = (
        data["GDU Lots Traded"]
        .round()
        .astype(int)
    )

    rows = []

    for gdu_lots, group in data.groupby(
        "GDU Lots Traded"
    ):

        total_trade_lots = group["Lots"].sum()
        total_pnl = group["PnL"].sum()

        rows.append({

            "GDU Lots Traded":
                gdu_lots,

            "Trades":
                len(group),

            "Total Trade Lots":
                total_trade_lots,

            "Total PnL":
                total_pnl,

            "Average PnL / Lot":
                total_pnl / total_trade_lots
                if total_trade_lots
                else np.nan,

            "Median PnL":
                group["PnL"].median(),

            # Pandas mean automatically ignores NaN.
            "Avg GC 10 Sec Range (Ticks)":
                group[
                    "GC 10 Sec Range (Ticks)"
                ].mean(),

            "Avg GC 1 Min Range (Ticks)":
                group[
                    "GC 1 Min Range (Ticks)"
                ].mean(),

            "Avg GC 5 Min Range (Ticks)":
                group[
                    "GC 5 Min Range (Ticks)"
                ].mean(),

            "Avg GC 30 Min Range (Ticks)":
                group[
                    "GC 30 Min Range (Ticks)"
                ].mean()
        })

    return pd.DataFrame(rows).sort_values(
        "GDU Lots Traded"
    )


# ============================================================
# HALF-HOUR ANALYSIS
# ============================================================

def create_half_hour(trades):

    df = trades.copy()

    df["Half Hour Start"] = (
        df["Entry Time"]
        .dt.floor("30min")
    )

    df["Half Hour End"] = (
        df["Half Hour Start"]
        + pd.Timedelta(minutes=30)
    )

    rows = []

    for start, group in df.groupby(
        "Half Hour Start"
    ):

        total_lots = group["Lots"].sum()
        pnl = group["PnL"].sum()

        rows.append({

            "Half Hour Start":
                start,

            "Half Hour End":
                start + pd.Timedelta(minutes=30),

            "PnL":
                pnl,

            "Number of Trades":
                len(group),

            "Lots":
                total_lots,

            "Average PnL / Lot":
                pnl / total_lots
                if total_lots
                else np.nan,

            "Avg GC 10 Sec Range (Ticks)":
                group[
                    "GC 10 Sec Range (Ticks)"
                ].mean(),

            "Avg GC 1 Min Range (Ticks)":
                group[
                    "GC 1 Min Range (Ticks)"
                ].mean(),

            "Avg GC 5 Min Range (Ticks)":
                group[
                    "GC 5 Min Range (Ticks)"
                ].mean(),

            "Avg GC 30 Min Range (Ticks)":
                group[
                    "GC 30 Min Range (Ticks)"
                ].mean()
        })

    return pd.DataFrame(rows)


# ============================================================
# CUMULATIVE HALF HOUR
# ============================================================

def create_cumulative_half_hour(trades):

    df = trades.copy()

    df["Bucket"] = (
        df["Entry Time"]
        .dt.floor("30min")
        .dt.strftime("%H:%M")
    )

    rows = []

    for bucket, group in df.groupby(
        "Bucket"
    ):

        total_lots = group["Lots"].sum()
        pnl = group["PnL"].sum()

        rows.append({

            "Time Bucket":
                bucket,

            "PnL":
                pnl,

            "Number of Trades":
                len(group),

            "Lots":
                total_lots,

            "Average PnL / Lot":
                pnl / total_lots
                if total_lots
                else np.nan,

            "Avg GC 10 Sec Range (Ticks)":
                group[
                    "GC 10 Sec Range (Ticks)"
                ].mean(),

            "Avg GC 1 Min Range (Ticks)":
                group[
                    "GC 1 Min Range (Ticks)"
                ].mean(),

            "Avg GC 5 Min Range (Ticks)":
                group[
                    "GC 5 Min Range (Ticks)"
                ].mean(),

            "Avg GC 30 Min Range (Ticks)":
                group[
                    "GC 30 Min Range (Ticks)"
                ].mean()
        })

    return pd.DataFrame(rows)


# ============================================================
# SUMMARY
# ============================================================

def create_summary(
    fills,
    trades
):

    wins = trades[
        trades["PnL"] > 0
    ]

    losses = trades[
        trades["PnL"] < 0
    ]

    flat = trades[
        trades["PnL"] == 0
    ]

    total_pnl = trades["PnL"].sum()
    total_lots = trades["Lots"].sum()

    summary = [

        ["Metric", "Value"],

        ["Raw fills",
         len(fills)],

        ["FIFO matched trades",
         len(trades)],

        ["Matched lots",
         total_lots],

        ["Total PnL",
         total_pnl],

        ["Average PnL / Trade",
         trades["PnL"].mean()],

        ["Average PnL / Lot",
         total_pnl / total_lots],

        ["Winning Trades",
         len(wins)],

        ["Losing Trades",
         len(losses)],

        ["Flat Trades",
         len(flat)],

        ["Win Rate",
         len(wins) / len(trades)
         if len(trades)
         else np.nan],

        ["Average Win",
         wins["PnL"].mean()
         if len(wins)
         else np.nan],

        ["Average Loss",
         losses["PnL"].mean()
         if len(losses)
         else np.nan],

        ["Trades with GC 10s Range",
         trades[
             "GC 10 Sec Range (Ticks)"
         ].notna().sum()],

        ["Trades with GC 1m Range",
         trades[
             "GC 1 Min Range (Ticks)"
         ].notna().sum()],

        ["Trades with GC 5m Range",
         trades[
             "GC 5 Min Range (Ticks)"
         ].notna().sum()],

        ["Trades with GC 30m Range",
         trades[
             "GC 30 Min Range (Ticks)"
         ].notna().sum()]
    ]

    return pd.DataFrame(
        summary[1:],
        columns=summary[0]
    )


# ============================================================
# PNL HISTOGRAM DATA
# ============================================================

def create_pnl_histogram(
    trades,
    bins=20
):

    pnl = trades["PnL"].dropna()

    counts, edges = np.histogram(
        pnl,
        bins=bins
    )

    return pd.DataFrame({

        "Bucket Start":
            edges[:-1],

        "Bucket End":
            edges[1:],

        "Count":
            counts
    })


# ============================================================
# MAIN
# ============================================================

def main():

    print("Loading fills...")

    fills = load_fills(
        FILLS_FILE
    )

    print(
        f"Raw fills: {len(fills)}"
    )

    # --------------------------------------------------------
    # FIFO
    # --------------------------------------------------------

    print("Running FIFO...")

    trades = fifo_match(
        fills
    )

    print(
        f"Matched trades: {len(trades)}"
    )

    # --------------------------------------------------------
    # Time & Sales
    # --------------------------------------------------------

    print(
        "Loading GDU Time & Sales..."
    )

    ts = load_time_sales(
        TS_FILE
    )

    # --------------------------------------------------------
    # GC
    # --------------------------------------------------------

    print(
        "Loading GC OHLC..."
    )

    gc_aug = load_gc_sheet(
        GC_FILE,
        "GC Aug26"
    )

    gc_dec = load_gc_sheet(
        GC_FILE,
        "GC Dec26"
    )

    # --------------------------------------------------------
    # Add GDU + GC information
    # --------------------------------------------------------

    print(
        "Calculating GDU lots and GC ranges..."
    )

    trades = enrich_trades(
        trades,
        ts,
        gc_aug,
        gc_dec
    )

    # --------------------------------------------------------
    # Analytics
    # --------------------------------------------------------

    summary = create_summary(
        fills,
        trades
    )

    gdu_analysis = (
        create_gdu_analysis(
            trades
        )
    )

    half_hour = (
        create_half_hour(
            trades
        )
    )

    cumulative = (
        create_cumulative_half_hour(
            trades
        )
    )

    histogram = (
        create_pnl_histogram(
            trades
        )
    )

    # --------------------------------------------------------
    # EXPORT
    # --------------------------------------------------------

    print(
        "Writing Excel..."
    )

    with pd.ExcelWriter(
        OUTPUT_FILE,
        engine="openpyxl"
    ) as writer:

        summary.to_excel(
            writer,
            sheet_name="Summary",
            index=False
        )

        trades.to_excel(
            writer,
            sheet_name="Trade Sheet",
            index=False
        )

        gdu_analysis.to_excel(
            writer,
            sheet_name="GDU Analysis",
            index=False
        )

        half_hour.to_excel(
            writer,
            sheet_name="Half Hour",
            index=False
        )

        cumulative.to_excel(
            writer,
            sheet_name="Cumulative Half Hour",
            index=False
        )

        histogram.to_excel(
            writer,
            sheet_name="PnL Histogram",
            index=False
        )

    print()
    print("DONE")
    print(
        Path(OUTPUT_FILE).resolve()
    )


if __name__ == "__main__":
    main()

Loading fills...
Fill columns:
['DATE', 'TIME', 'EXCH', 'CONTRACT', 'B/S', 'QTY', 'PRICE']
Raw fills: 442
Running FIFO...
Matched trades: 221
Loading GDU Time & Sales...
Loading GC OHLC...


/var/folders/mx/3q6k9m2j38j96jc_zrts6lq00000gn/T/ipykernel_41624/2674334333.py:45: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  date_text = pd.to_datetime(date_col).dt.strftime("%Y-%m-%d")


Calculating GDU lots and GC ranges...
Writing Excel...

DONE
/Users/adityagiri/Axxela/GDU GC/GDU_Analysis_FIFO_GC_Ranges_DivideBy0.1.xlsx


In [17]:
"""
Same treatment as GDU_Analysis_FIFO_GC_Ranges_DivideBy0.1_modified.xlsx, applied
to GDU_Analysis_FIFO_GC_Ranges_DivideBy0.1.xlsx. This workbook is a different
(newer, 222-trade, all GDU Dec26) dataset and - unlike the "_modified" file -
does NOT already have Profit / loss Making / Flat Trades split sheets, so
this script builds those first (split off Trade Sheet by PnL sign, same
columns plus a computed "Time in sec" = Duration (ms) / 1000, since this
Trade Sheet only has Duration (ms), not Time in sec), then adds the same
"Trades & Lots by 1-Second Duration Bucket" table + chart to each, and
rebuilds PnL Histogram the same way (round bins, red/green/gray coloring).

Bin width for the PnL histogram is chosen adaptively (smallest "nice" width
from a candidate list that keeps the bin count around 20-30) since this
dataset's PnL range (-5,610 to +1,620) is much wider than the previous
file's (-790 to +480) - a fixed $50 width would produce ~150 mostly-empty
bins here.
"""
import math
from collections import defaultdict

import openpyxl
from openpyxl.chart import BarChart, Reference
from openpyxl.chart.marker import DataPoint
from openpyxl.chart.shapes import GraphicalProperties
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

SRC = "GDU_Analysis_FIFO_GC_Ranges_DivideBy0.1.xlsx"
OUT = "GDU_Analysis_FIFO_GC_Ranges_DivideBy0.1 (Histograms Added).xlsx"

HEADER_FONT = Font(bold=True, color="FFFFFFFF")
HEADER_FILL = PatternFill("solid", fgColor="FF1F4E78")
HEADER_ALIGN = Alignment(horizontal="center", vertical="center")

GREEN = "FF548235"
RED = "FFC00000"
GRAY = "FF808080"

wb = openpyxl.load_workbook(SRC)
ts_ws = wb["Trade Sheet"]

TS_HEADERS = [ts_ws.cell(row=1, column=c).value for c in range(1, ts_ws.max_column + 1)]
PNL_COL = TS_HEADERS.index("PnL") + 1
DUR_COL = TS_HEADERS.index("Duration (ms)") + 1
LOTS_COL = TS_HEADERS.index("Lots") + 1
TIME_SEC_COL = len(TS_HEADERS) + 1  # new column appended after the last existing one

# ---------------------------------------------------------------------------
# 0. Split Trade Sheet into Profit / loss Making / Flat Trades (this workbook
#    doesn't have them yet, unlike the "_modified" file), adding a computed
#    "Time in sec" column since this Trade Sheet only has Duration (ms).
# ---------------------------------------------------------------------------
rows_by_cat = {"Profit": [], "loss Making": [], "Flat Trades": []}
for r in range(2, ts_ws.max_row + 1):
    pnl = ts_ws.cell(row=r, column=PNL_COL).value
    if pnl is None:
        continue
    values = [ts_ws.cell(row=r, column=c).value for c in range(1, len(TS_HEADERS) + 1)]
    dur_ms = ts_ws.cell(row=r, column=DUR_COL).value
    time_sec = (dur_ms / 1000.0) if dur_ms is not None else None
    values.append(time_sec)
    if pnl > 0:
        rows_by_cat["Profit"].append(values)
    elif pnl < 0:
        rows_by_cat["loss Making"].append(values)
    else:
        rows_by_cat["Flat Trades"].append(values)

print("Split counts:", {k: len(v) for k, v in rows_by_cat.items()})

full_headers = TS_HEADERS + ["Time in sec"]
for cat_name, rows in rows_by_cat.items():
    ws = wb.create_sheet(cat_name)
    for j, h in enumerate(full_headers, start=1):
        c = ws.cell(row=1, column=j, value=h)
        c.font = HEADER_FONT
        c.fill = HEADER_FILL
        c.alignment = HEADER_ALIGN
    for i, values in enumerate(rows):
        r = 2 + i
        for j, v in enumerate(values, start=1):
            ws.cell(row=r, column=j, value=v)
    for j in range(1, len(full_headers) + 1):
        ws.column_dimensions[get_column_letter(j)].width = 16

# ---------------------------------------------------------------------------
# 1. Trades & Lots by 1-second duration bucket - Profit / loss Making / Flat Trades
#    (identical logic/layout to the "_modified" file build; none of these
#    sheets pre-existed here so there's no old chart/table to overwrite)
# ---------------------------------------------------------------------------
sheet_layout = {
    "Profit": (19, 20, 21),        # S, T, U
    "loss Making": (19, 20, 21),   # S, T, U
    "Flat Trades": (19, 20, 21),   # S, T, U
}

for name, (bcol, tcol, lcol) in sheet_layout.items():
    ws = wb[name]
    n_rows = ws.max_row

    buckets = defaultdict(lambda: [0, 0])
    for r in range(2, n_rows + 1):
        t = ws.cell(row=r, column=TIME_SEC_COL).value
        lots = ws.cell(row=r, column=LOTS_COL).value
        if t is None or lots is None:
            continue
        b = int(math.floor(t))
        buckets[b][0] += 1
        buckets[b][1] += lots

    bucket_starts = sorted(buckets.keys())
    print(f"{name}: {len(bucket_starts)} non-empty 1-sec buckets, "
          f"trades={sum(v[0] for v in buckets.values())}, "
          f"lots={sum(v[1] for v in buckets.values())}")

    headers = ["Bucket", "Trade Count", "Total Lots"]
    for col, h in zip((bcol, tcol, lcol), headers):
        c = ws.cell(row=1, column=col, value=h)
        c.font = HEADER_FONT
        c.fill = HEADER_FILL
        c.alignment = HEADER_ALIGN

    for i, b in enumerate(bucket_starts):
        r = 2 + i
        ws.cell(row=r, column=bcol, value=f"{b}-{b + 1}")
        ws.cell(row=r, column=tcol, value=buckets[b][0])
        ws.cell(row=r, column=lcol, value=int(buckets[b][1]))

    last_row = 1 + len(bucket_starts)
    ws.column_dimensions[get_column_letter(bcol)].width = 12
    ws.column_dimensions[get_column_letter(tcol)].width = 14
    ws.column_dimensions[get_column_letter(lcol)].width = 14

    chart = BarChart()
    chart.type = "col"
    chart.grouping = "clustered"
    chart.gapWidth = 60
    chart.title = "Trades & Lots by 1-Second Duration Bucket"
    chart.y_axis.title = "Count"
    chart.x_axis.title = "Duration Bucket (sec)"
    chart.height = 9
    chart.width = 20

    cats = Reference(ws, min_col=bcol, min_row=2, max_row=last_row)
    data_trades = Reference(ws, min_col=tcol, min_row=1, max_row=last_row)
    data_lots = Reference(ws, min_col=lcol, min_row=1, max_row=last_row)
    chart.add_data(data_trades, titles_from_data=True)
    chart.add_data(data_lots, titles_from_data=True)
    chart.set_categories(cats)

    anchor_col_letter = get_column_letter(lcol + 2)
    ws.add_chart(chart, f"{anchor_col_letter}2")

# ---------------------------------------------------------------------------
# 2. PnL Histogram - rebuild with round bins (width chosen adaptively to
#    target ~20-30 bins) + red/green/gray coloring
# ---------------------------------------------------------------------------
pnls = [ts_ws.cell(row=r, column=PNL_COL).value for r in range(2, ts_ws.max_row + 1)]
pnls = [v for v in pnls if v is not None]

CANDIDATE_WIDTHS = [10, 20, 25, 50, 100, 200, 250, 500, 1000, 2000, 2500, 5000]
data_range = max(pnls) - min(pnls)
BIN_WIDTH = CANDIDATE_WIDTHS[-1]
for w in CANDIDATE_WIDTHS:
    if data_range / w <= 30:
        BIN_WIDTH = w
        break

lo = math.floor(min(pnls) / BIN_WIDTH) * BIN_WIDTH
hi = math.ceil(max(pnls) / BIN_WIDTH) * BIN_WIDTH
n_bins = int(round((hi - lo) / BIN_WIDTH))

bin_counts = [0] * n_bins
for v in pnls:
    idx = int((v - lo) // BIN_WIDTH)
    idx = min(max(idx, 0), n_bins - 1)
    bin_counts[idx] += 1

print(f"PnL Histogram: bin width {BIN_WIDTH}, {n_bins} bins, range [{lo}, {hi}], "
      f"total count {sum(bin_counts)} (should equal {len(pnls)})")

hist_ws = wb["PnL Histogram"]
hist_ws._charts = []
for r in range(1, max(hist_ws.max_row, n_bins + 1) + 5):
    for c in range(1, 6):
        hist_ws.cell(row=r, column=c, value=None)

headers = ["Bucket Start", "Bucket End", "Bucket Label", "Count"]
for j, h in enumerate(headers, start=1):
    c = hist_ws.cell(row=1, column=j, value=h)
    c.font = HEADER_FONT
    c.fill = HEADER_FILL
    c.alignment = HEADER_ALIGN

for i in range(n_bins):
    r = 2 + i
    b_start = lo + i * BIN_WIDTH
    b_end = b_start + BIN_WIDTH
    hist_ws.cell(row=r, column=1, value=b_start)
    hist_ws.cell(row=r, column=2, value=b_end)
    hist_ws.cell(row=r, column=3, value=f"{b_start:g} to {b_end:g}")
    hist_ws.cell(row=r, column=4, value=bin_counts[i])

last_row = 1 + n_bins
hist_ws.column_dimensions["A"].width = 13
hist_ws.column_dimensions["B"].width = 13
hist_ws.column_dimensions["C"].width = 18
hist_ws.column_dimensions["D"].width = 10

chart = BarChart()
chart.type = "col"
chart.grouping = "clustered"
chart.gapWidth = 0
chart.title = f"PnL Distribution ({len(pnls)} Trades, ${BIN_WIDTH} Buckets)"
chart.y_axis.title = "Number of Trades"
chart.x_axis.title = "PnL Bucket ($)"
chart.height = 10
chart.width = 24
chart.legend = None

cats = Reference(hist_ws, min_col=3, min_row=2, max_row=last_row)
data = Reference(hist_ws, min_col=4, min_row=1, max_row=last_row)
chart.add_data(data, titles_from_data=True)
chart.set_categories(cats)

series = chart.series[0]
series.graphicalProperties.solidFill = "FF4472C4"

data_points = []
for i in range(n_bins):
    b_start = lo + i * BIN_WIDTH
    b_end = b_start + BIN_WIDTH
    if b_end <= 0:
        color = RED
    elif b_start >= 0:
        color = GREEN
    else:
        color = GRAY
    dp = DataPoint(idx=i)
    dp.graphicalProperties = GraphicalProperties(solidFill=color)
    data_points.append(dp)
series.data_points = data_points

hist_ws.add_chart(chart, "F2")

wb.save(OUT)
print(f"Saved {OUT}")

Split counts: {'Profit': 17, 'loss Making': 161, 'Flat Trades': 43}
Profit: 4 non-empty 1-sec buckets, trades=17, lots=49
loss Making: 6 non-empty 1-sec buckets, trades=161, lots=427
Flat Trades: 2 non-empty 1-sec buckets, trades=43, lots=86
PnL Histogram: bin width 20, 27 bins, range [-420, 120], total count 221 (should equal 221)
Saved GDU_Analysis_FIFO_GC_Ranges_DivideBy0.1 (Histograms Added).xlsx
